In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.extend(['../', '../Benchmark/', '../Stretch2Relax', '../curved_linesearch'])
import MeshFEM
import mesh, mesh_energy
import energy
import parametrization, viewer, benchmark, py_newton_optimizer
import flip_avoiding_step_length
import numpy as np

In [ ]:
import param_utils
import helper_funcs

# Load Mesh

In [ ]:
mesh_path = '../../models/hilbert_curve_small.msh.xz'

In [ ]:
m = helper_funcs.read_mesh(mesh_path)

In [ ]:
print(f'mesh vertices: {m.numVertices()}; mesh elements: {m.numElements()}')

# Problem Setup

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
# mesh initialization
# uv_init = parametrization.lscm(m)

bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = parametrization.harmonic(m, bdry_uv)
# uv_init = parametrization.lscm(m)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)

uv.setVars(uv_init.ravel())

In [ ]:
psi = energy.SymmetricDirichlet(2)

methods = {
    'newton': mesh_energy.Parametrization,
    'proj_rest': mesh_energy.ParametrizationProjectToRestHessian,
    'akvf': mesh_energy.ParametrizationAKVF
}

In [ ]:
method = 'akvf'
hpc = py_newton_optimizer.HessianProjectionAdaptive()
hpc.numConsecutiveIndefiniteStepsBeforeEnable = 0
hpc.numProjectionStepsBeforeDisable = 1
hpc.startWithProjectionActive = False

hpc = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
param = methods[method](m, uv, psi)

# UV viewer

In [ ]:
uv_viewer = viewer.Viewer(MeshFEM.EmbeddedMesh(m, uv), wireframe=True)
uv_viewer.show()

In [ ]:
uv_viewer.setCameraParams(((0.3124286426750361, 0.006886896732418157, 9.246767259673106),
 (0.0, 1.0, 0.0),
 (0.3124286426750361, 0.006886896732418157, 0.0)))

# Initialization

In [ ]:
# uv.setVars(np.load('init.npy'))

In [ ]:
import initial_utils
# scale = initial_utils.initialization_scale(m, uv, param, 'grad_minimal')
# uv.setVars(scale * uv_init.ravel())

# Optimize

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])

# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-12
opt = prob.optimizer()

In [ ]:
prob.useRelativeHessianShift = False

In [ ]:
# Applying flip-avoiding linesearch
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8

In [ ]:
opt.options.verboseNonPosDef = True
opt.options.niter = 200
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()

In [ ]:
def cb(prob, it):
    uv_2d = uv.getVars().reshape(-1, 2)
    uv_2d -= np.mean(uv_2d, axis=0)
    uv.setVars(uv_2d.ravel())
    uv_viewer.update()
prob.setCustomIterationCallback(cb)

In [ ]:
hpc_name = hpc.__class__.__name__
vid_path = f'hilbert_curve_{method}_{hpc.__class__.__name__}.mp4'

In [ ]:
benchmark.reset()
uv_viewer.recordStart(vid_path, outputScale=2, renderScale=4, lineWidthScale=0.5)
cr = opt.optimize()
uv_viewer.recordStop()
benchmark.report()